# Пошаговая локализация: CLAHE → template match → SIFT/ORB → RANSAC

Пример: `examples/case_01_100m`.

Нужен интернет (или тёплый `.cache/tiles`) — качаем Esri World Imagery вокруг prior-координат.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from app.georeferencer import Georeferencer
from app.localizer import Localizer
from app.map_fetcher import MapFetcher
from app.models import LocalizationRequest

CASE = ROOT / "examples" / "case_01_100m"
OUT = ROOT / "notebooks" / "artifacts" / "case_01_100m"
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["image.cmap"] = "gray"


def show_bgr(img, title="", ax=None):
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img
    if ax is None:
        plt.figure()
        plt.imshow(rgb)
        plt.title(title)
        plt.axis("off")
        plt.show()
    else:
        ax.imshow(rgb)
        ax.set_title(title)
        ax.axis("off")


print("CASE:", CASE)
print("OUT:", OUT)

## 1. Загрузка query + meta

In [ ]:
meta_raw = json.loads((CASE / "meta.json").read_text(encoding="utf-8"))
expected = json.loads((CASE / "expected.json").read_text(encoding="utf-8"))
query_bgr = cv2.imread(str(CASE / "query.jpg"))
assert query_bgr is not None, "не удалось прочитать query.jpg"

h, w = query_bgr.shape[:2]
req = LocalizationRequest(**meta_raw)
print("meta:", meta_raw)
print("expected GT lat/lon:", expected["lat_deg"], expected["lon_deg"])
print("query size:", w, "x", h)

show_bgr(query_bgr, "query.jpg (имитация снимка БПЛА)")

## 2. Georeferencer → план поиска + загрузка карты

Считаем GSD/zoom и качаем тайлы вокруг prior (`lat/lon` из meta, не GT).

In [ ]:
plan = Georeferencer().plan(req, (w, h))
print(
    f"GSD≈{plan.gsd_m:.3f} м/px | zoom={plan.map_zoom} | "
    f"radius={plan.search_radius_m} м | yaw={plan.yaw_deg}°"
)

fetcher = MapFetcher(cache_dir=ROOT / ".cache" / "tiles")
map_raster = fetcher.fetch(plan)
map_bgr = map_raster.image
print(f"map: {map_raster.width}x{map_raster.height} zoom={map_raster.zoom} source={map_raster.source}")

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
show_bgr(query_bgr, "query", ax=ax[0])
show_bgr(map_bgr, "map around prior", ax=ax[1])
plt.tight_layout()
plt.show()

cv2.imwrite(str(OUT / "00_map.jpg"), map_bgr)

## 3. Поворот query на `-yaw` (north-up)

In [ ]:
loc = Localizer()
query_rot, rot_m = loc._rotate_image(query_bgr, -plan.yaw_deg)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
show_bgr(query_bgr, f"до поворота (yaw={plan.yaw_deg}°)", ax=ax[0])
show_bgr(query_rot, f"после поворота на {-plan.yaw_deg}°", ax=ax[1])
plt.tight_layout()
plt.show()

cv2.imwrite(str(OUT / "01_query_rotated.jpg"), query_rot)

## 4. CLAHE

Grayscale + локальное выравнивание контраста — и для query, и для карты.

In [ ]:
map_work, map_scale = loc._downscale(map_bgr, loc.max_map_side)
query_work, _ = loc._downscale(query_rot, min(loc.max_map_side, max(query_rot.shape[:2])))

q_gray = cv2.cvtColor(query_work, cv2.COLOR_BGR2GRAY)
m_gray = cv2.cvtColor(map_work, cv2.COLOR_BGR2GRAY)
q_clahe = loc._preprocess(query_work)
m_clahe = loc._preprocess(map_work)

fig, ax = plt.subplots(2, 2, figsize=(12, 10))
ax[0, 0].imshow(q_gray, cmap="gray")
ax[0, 0].set_title("query gray")
ax[0, 1].imshow(q_clahe, cmap="gray")
ax[0, 1].set_title("query CLAHE")
ax[1, 0].imshow(m_gray, cmap="gray")
ax[1, 0].set_title("map gray")
ax[1, 1].imshow(m_clahe, cmap="gray")
ax[1, 1].set_title("map CLAHE")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()

cv2.imwrite(str(OUT / "02_query_clahe.jpg"), q_clahe)
cv2.imwrite(str(OUT / "02_map_clahe.jpg"), m_clahe)
print(f"map_work scale={map_scale:.3f}, size={map_work.shape[1]}x{map_work.shape[0]}")

## 5. Coarse: `matchTemplate`

Ищем, где на карте query лучше всего совпадает по яркости. Рисуем heatmap score и ROI.

In [ ]:
# Повторяем логику coarse вручную, чтобы показать heatmap.
m_small, m_s = loc._downscale_gray(m_clahe, loc.coarse_max_side)
q_target = min(loc.coarse_max_side, max(64, int(max(q_clahe.shape) * m_s)))
q_small, _ = loc._downscale_gray(q_clahe, q_target)

if q_small.shape[0] >= m_small.shape[0] or q_small.shape[1] >= m_small.shape[1]:
    scale = min(
        (m_small.shape[0] * 0.45) / max(q_small.shape[0], 1),
        (m_small.shape[1] * 0.45) / max(q_small.shape[1], 1),
    )
    q_small = cv2.resize(
        q_small,
        (max(8, int(q_small.shape[1] * scale)), max(8, int(q_small.shape[0] * scale))),
    )

heat = cv2.matchTemplate(m_small, q_small, cv2.TM_CCOEFF_NORMED)
_, max_val, _, max_loc = cv2.minMaxLoc(heat)

coarse = loc._coarse_locate(query_work, map_work)
assert coarse is not None, "coarse failed"
cx, cy, cw, ch, coarse_score = coarse
print(f"coarse_score={coarse_score:.3f}  ROI=({cx},{cy},{cw},{ch})  heatmap_peak={max_val:.3f}")

if coarse_score >= 0.25:
    pad = int(0.4 * max(cw, ch))
    x0 = max(0, cx - pad)
    y0 = max(0, cy - pad)
    x1 = min(map_work.shape[1], cx + cw + pad)
    y1 = min(map_work.shape[0], cy + ch + pad)
    roi = map_work[y0:y1, x0:x1]
else:
    x0, y0 = 0, 0
    x1, y1 = map_work.shape[1], map_work.shape[0]
    roi = map_work
    print("coarse weak → fine on full map")

map_vis = map_work.copy()
cv2.rectangle(map_vis, (cx, cy), (cx + cw, cy + ch), (0, 255, 0), 2)
cv2.rectangle(map_vis, (x0, y0), (x1, y1), (0, 165, 255), 2)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(heat, cmap="magma")
ax[0].plot(max_loc[0], max_loc[1], "c+", markersize=14, markeredgewidth=2)
ax[0].set_title("template-match heatmap")
ax[0].axis("off")
show_bgr(map_vis, "зелёный=best match, оранж=ROI fine", ax=ax[1])
show_bgr(roi, "ROI для fine", ax=ax[2])
plt.tight_layout()
plt.show()

cv2.imwrite(str(OUT / "03_coarse_roi.jpg"), map_vis)
cv2.imwrite(str(OUT / "03_roi.jpg"), roi)

## 6. Ключевые точки (keypoints) — что это такое

**Ключевая точка** — маленький «якорь» на картинке: место, которое алгоритм считает узнаваемым (угол здания, пересечение дорог, контрастный blob).

У каждой точки есть:

| Поле | Смысл |
|------|--------|
| `(x, y)` | где на изображении |
| `size` | размер окрестности (масштаб) |
| `angle` | ориентация (куда «смотрит» патч) |
| **дескриптор** | числовой отпечаток окрестности (SIFT ≈ 128 float, ORB ≈ 256 бит) |

**Зачем:** сравнивать не картинки целиком, а пары «похожих якорей» на снимке и на карте. Если много якорей согласованно совпали — нашли, куда кадр ложится.

**Где появляются:** на резких углах и текстуре; почти не появляются на однотонном поле/воде.

Дальше: обзор всех точек → **увеличенный кусок**, чтобы было видно size/angle → matching по Lowe.

In [ ]:
q_f = loc._preprocess(query_work)
r_f = loc._preprocess(roi)
kp_q, des_q, backend = loc._detect_describe(q_f)
kp_r, des_r, _ = loc._detect_describe(r_f)
print(f"backend={backend}  kp_query={len(kp_q)}  kp_roi={len(kp_r)}")
print(f"descriptor shape query: {None if des_q is None else des_q.shape}  dtype={None if des_q is None else des_q.dtype}")

# Пример одной точки: координаты, размер, угол, кусок дескриптора
k0 = kp_q[0]
print(
    f"пример kp[0]: x={k0.pt[0]:.1f} y={k0.pt[1]:.1f} "
    f"size={k0.size:.1f}px  angle={k0.angle:.1f}°  response={k0.response:.3f}"
)
if des_q is not None:
    print(f"первые 8 чисел дескриптора: {np.array(des_q[0][:8]).round(2)}")

kp_q_img = cv2.drawKeypoints(
    query_work, kp_q, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)
kp_r_img = cv2.drawKeypoints(
    roi, kp_r, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
show_bgr(kp_q_img, f"все keypoints на query ({backend}, n={len(kp_q)})", ax=ax[0])
show_bgr(kp_r_img, f"все keypoints на ROI карты ({backend}, n={len(kp_r)})", ax=ax[1])
plt.tight_layout()
plt.show()

cv2.imwrite(str(OUT / "04_keypoints_query.jpg"), kp_q_img)
cv2.imwrite(str(OUT / "04_keypoints_roi.jpg"), kp_r_img)

### Увеличенный кусок с ключевыми точками

На полном кадре кружки сливаются. Ниже:
1. зелёная рамка — область zoom;
2. тот же кусок, увеличенный ×4, с кругами (`size`) и стрелками ориентации (`angle`);
3. несколько патчей 64×64 вокруг самых «сильных» точек — именно из такого патча считается дескриптор.

In [ ]:
def keypoints_in_roi(keypoints, x0, y0, x1, y1):
    """Keypoints whose centers fall inside [x0:x1, y0:y1]."""
    out = []
    for kp in keypoints:
        x, y = kp.pt
        if x0 <= x < x1 and y0 <= y < y1:
            out.append(kp)
    return out


def draw_keypoints_zoom(bgr, keypoints, x0, y0, x1, y1, scale=4):
    """Crop → upscale → draw size circles + orientation arrows."""
    crop = bgr[y0:y1, x0:x1].copy()
    big = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_NEAREST)
    local = []
    for kp in keypoints_in_roi(keypoints, x0, y0, x1, y1):
        lk = cv2.KeyPoint(
            (kp.pt[0] - x0) * scale,
            (kp.pt[1] - y0) * scale,
            max(kp.size * scale, 8.0),
            kp.angle,
            kp.response,
            kp.octave,
            kp.class_id,
        )
        local.append(lk)

    vis = cv2.drawKeypoints(
        big, local, None, color=(0, 255, 255),
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
    )
    # Толстые маркеры поверх — лучше видно на презентации
    for lk in local:
        cx, cy = int(round(lk.pt[0])), int(round(lk.pt[1]))
        rad = max(4, int(round(lk.size / 2)))
        cv2.circle(vis, (cx, cy), rad, (0, 255, 0), 2, cv2.LINE_AA)
        cv2.circle(vis, (cx, cy), 2, (0, 0, 255), -1, cv2.LINE_AA)
        if lk.angle >= 0:
            ang = np.deg2rad(lk.angle)
            ex = int(round(cx + rad * np.cos(ang)))
            ey = int(round(cy + rad * np.sin(ang)))
            cv2.arrowedLine(vis, (cx, cy), (ex, ey), (255, 128, 0), 2, tipLength=0.35)
    return vis, local


# Берём окно вокруг самой «сильной» точки (высокий response), с запасом.
best = max(kp_q, key=lambda k: k.response)
bx, by = int(best.pt[0]), int(best.pt[1])
zoom_w, zoom_h = 140, 140
hq, wq = query_work.shape[:2]
zx0 = int(np.clip(bx - zoom_w // 2, 0, max(0, wq - zoom_w)))
zy0 = int(np.clip(by - zoom_h // 2, 0, max(0, hq - zoom_h)))
zx1, zy1 = zx0 + zoom_w, zy0 + zoom_h

overview = query_work.copy()
cv2.rectangle(overview, (zx0, zy0), (zx1, zy1), (0, 255, 0), 2)
cv2.circle(overview, (bx, by), 6, (0, 0, 255), 2)

zoom_vis, local_kps = draw_keypoints_zoom(query_work, kp_q, zx0, zy0, zx1, zy1, scale=4)
print(f"zoom window=({zx0},{zy0})-({zx1},{zy1})  точек внутри: {len(local_kps)}")
print(f"якорь: response={best.response:.3f} size={best.size:.1f} angle={best.angle:.1f}°")

fig, ax = plt.subplots(1, 2, figsize=(14, 7))
show_bgr(overview, "query: зелёная рамка = область zoom", ax=ax[0])
show_bgr(
    zoom_vis,
    f"zoom ×4: круг=size, стрелка=angle, n={len(local_kps)}",
    ax=ax[1],
)
plt.tight_layout()
plt.show()

# Несколько патчей вокруг топ-точек по response
top = sorted(kp_q, key=lambda k: k.response, reverse=True)[:6]
patch = 32
fig, ax = plt.subplots(2, 6, figsize=(14, 5))
for i, kp in enumerate(top):
    x, y = int(round(kp.pt[0])), int(round(kp.pt[1]))
    x0p, y0p = max(0, x - patch), max(0, y - patch)
    x1p, y1p = min(wq, x + patch), min(hq, y + patch)
    crop = query_work[y0p:y1p, x0p:x1p]
    crop_big = cv2.resize(crop, (128, 128), interpolation=cv2.INTER_NEAREST)
    # отметить центр
    c = crop_big.copy()
    cv2.line(c, (56, 64), (72, 64), (0, 255, 0), 1, cv2.LINE_AA)
    cv2.line(c, (64, 56), (64, 72), (0, 255, 0), 1, cv2.LINE_AA)
    show_bgr(c, f"#{i} r={kp.response:.2f}", ax=ax[0, i])

    # тот же патч с CLAHE — что реально видит дескриптор
    g = q_f[y0p:y1p, x0p:x1p]
    g_big = cv2.resize(g, (128, 128), interpolation=cv2.INTER_NEAREST)
    ax[1, i].imshow(g_big, cmap="gray")
    ax[1, i].set_title(f"CLAHE  ang={kp.angle:.0f}°")
    ax[1, i].axis("off")

ax[0, 0].set_ylabel("BGR patch")
ax[1, 0].set_ylabel("CLAHE patch")
plt.suptitle("Патчи вокруг самых сильных keypoints (из них считают дескриптор)")
plt.tight_layout()
plt.show()

cv2.imwrite(str(OUT / "04b_keypoints_overview_box.jpg"), overview)
cv2.imwrite(str(OUT / "04b_keypoints_zoom.jpg"), zoom_vis)

### Matching дескрипторов (Lowe ratio)

Для каждой точки query ищем 2 ближайших дескриптора на карте. Match оставляем только если лучший заметно лучше второго (`dist1 < 0.75 × dist2`) — иначе точка «похожа сразу на две» и ненадёжна.

In [ ]:
norm = cv2.NORM_L2 if backend == "sift" else cv2.NORM_HAMMING
matcher = cv2.BFMatcher(norm)
knn = matcher.knnMatch(des_q, des_r, k=2)

good = []
for pair in knn:
    if len(pair) < 2:
        continue
    m, n = pair
    if m.distance < 0.75 * n.distance:
        good.append(m)
print(f"good matches after Lowe: {len(good)}  (из {len(kp_q)} точек query)")

draw_n = min(80, len(good))
match_img = cv2.drawMatches(
    query_work,
    kp_q,
    roi,
    kp_r,
    good[:draw_n],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)
show_bgr(match_img, f"Lowe matches (показываем {draw_n} из {len(good)})")
cv2.imwrite(str(OUT / "04_matches_lowe.jpg"), match_img)

## 7. RANSAC → affine + inliers

Оставляем только геометрически согласованные соответствия и переносим центр кадра на карту.

In [ ]:
src = np.float32([kp_q[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
dst = np.float32([kp_r[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

M, mask = cv2.estimateAffinePartial2D(
    src,
    dst,
    method=cv2.RANSAC,
    ransacReprojThreshold=5.0,
    maxIters=4000,
    confidence=0.99,
)
assert M is not None and mask is not None, "RANSAC failed"

inlier_mask = mask.ravel().astype(bool)
inliers = int(inlier_mask.sum())
inlier_matches = [m for m, keep in zip(good, inlier_mask) if keep]
outlier_matches = [m for m, keep in zip(good, inlier_mask) if not keep]

inlier_src = src[inlier_mask]
inlier_dst = dst[inlier_mask]
pred = cv2.transform(inlier_src, M)
residual = float(np.linalg.norm(pred - inlier_dst, axis=2).mean())
affine_scale = float(np.hypot(M[0, 0], M[0, 1]))

print(f"inliers={inliers}/{len(good)}  residual={residual:.2f}px  scale={affine_scale:.3f}")
print("affine M:\n", M)

inlier_img = cv2.drawMatches(
    query_work,
    kp_q,
    roi,
    kp_r,
    inlier_matches[:100],
    None,
    matchColor=(0, 255, 0),
    singlePointColor=(200, 200, 200),
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)
outlier_img = cv2.drawMatches(
    query_work,
    kp_q,
    roi,
    kp_r,
    outlier_matches[:60],
    None,
    matchColor=(0, 0, 255),
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)

fig, ax = plt.subplots(2, 1, figsize=(14, 10))
show_bgr(inlier_img, f"RANSAC inliers (зелёные), n={inliers}", ax=ax[0])
show_bgr(outlier_img, f"outliers (красные), n={len(outlier_matches)}", ax=ax[1])
plt.tight_layout()
plt.show()

cv2.imwrite(str(OUT / "05_inliers.jpg"), inlier_img)
cv2.imwrite(str(OUT / "05_outliers.jpg"), outlier_img)

## 8. Центр кадра → lat/lon + сравнение с GT

In [ ]:
qcx = query_work.shape[1] / 2.0
qcy = query_work.shape[0] / 2.0
mapped = cv2.transform(np.array([[[qcx, qcy]]], dtype=np.float32), M)[0, 0]
mx = (mapped[0] + x0) / map_scale
my = (mapped[1] + y0) / map_scale
lat, lon = map_raster.pixel_to_latlon(mx, my)

gt_lat, gt_lon = expected["lat_deg"], expected["lon_deg"]
# грубая оценка ошибки в метрах
err_m = np.hypot(
    (lat - gt_lat) * 111_320.0,
    (lon - gt_lon) * 111_320.0 * np.cos(np.radians(gt_lat)),
)

print(f"prior:     {req.lat_deg:.6f}, {req.lon_deg:.6f}")
print(f"estimated: {lat:.6f}, {lon:.6f}")
print(f"GT:        {gt_lat:.6f}, {gt_lon:.6f}")
print(f"error vs GT ≈ {err_m:.1f} m")

# Визуализация на полной карте
full = map_bgr.copy()
px_est = map_raster.latlon_to_pixel(lat, lon)
px_gt = map_raster.latlon_to_pixel(gt_lat, gt_lon)
px_prior = map_raster.latlon_to_pixel(req.lat_deg, req.lon_deg)


def cross(img, pt, color, label):
    x = int(round(float(pt[0])))
    y = int(round(float(pt[1])))
    cv2.line(img, (x - 12, y), (x + 12, y), color, 2, cv2.LINE_AA)
    cv2.line(img, (x, y - 12), (x, y + 12), color, 2, cv2.LINE_AA)
    cv2.putText(img, label, (x + 8, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)


cross(full, px_prior, (255, 128, 0), "prior")
cross(full, px_est, (0, 255, 0), "est")
cross(full, px_gt, (0, 0, 255), "GT")
cv2.arrowedLine(
    full,
    (int(px_prior[0]), int(px_prior[1])),
    (int(px_est[0]), int(px_est[1])),
    (0, 255, 255),
    2,
    tipLength=0.03,
)

show_bgr(full, "карта: prior → estimate, рядом GT")
cv2.imwrite(str(OUT / "06_result_on_map.jpg"), full)
print("артефакты:", OUT)

## 9. Тот же путь через готовый `Localizer` (контроль)

In [ ]:
result = loc.localize(query_bgr, map_raster, plan)
print("Localizer result:")
print(f"  lat/lon = {result.lat_deg:.6f}, {result.lon_deg:.6f}")
print(f"  confidence = {result.confidence:.3f}")
print("  diagnostics:")
for k, v in result.diagnostics.items():
    if k == "rotation_matrix":
        continue
    print(f"    {k}: {v}")